# TFM · Predicción de Consumo y Precio Eléctrico — Madrid
## Notebook 03 · Feature Engineering

**Objetivo:** Construir el dataset final listo para modelado a partir del dataset base.

**Variables target:** `precio_eur_mwh` y `demanda_mw` (modelos independientes)

**Horizonte de predicción:** 48 horas

**Secciones:**
1. Carga y revisión del dataset base
2. Features de lags (valores pasados)
3. Features de ventanas móviles (rolling)
4. Features de diferencias
5. Features de interacción
6. Construcción de targets multihorizonte (t+1 … t+48)
7. Escalado y split temporal train/val/test
8. Resumen y guardado

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize':    (14, 4),
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})
COLORS = ['#2563EB', '#F59E0B', '#10B981', '#EF4444']

Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('data/models').mkdir(parents=True, exist_ok=True)
print('✅ Setup completado')

---
## 1. Carga y revisión del dataset base

In [ ]:
df = pd.read_csv('data/processed/dataset_base.csv')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Madrid')
df = df.set_index('datetime').sort_index()

# Convertir booleanos si vienen como string
if df['es_festivo'].dtype == object:
    df['es_festivo'] = df['es_festivo'].map({'True': 1, 'False': 0}).astype(int)
else:
    df['es_festivo'] = df['es_festivo'].astype(int)

print(f'✅ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'   Período: {df.index.min().date()} → {df.index.max().date()}')
print(f'\nColumnas disponibles:')
print(df.dtypes)

---
## 2. Features de lags

Los lags son los valores pasados de las variables target. Son las features más importantes
para series temporales — el precio de ayer a la misma hora es el mejor predictor del precio de hoy.

Lags seleccionados según el análisis de autocorrelación del EDA:
- **t-1**: hora anterior
- **t-2, t-3**: inercia a corto plazo
- **t-24**: mismo momento del día anterior
- **t-48**: mismo momento de hace 2 días
- **t-168**: mismo momento de la semana anterior

In [ ]:
df_feat = df.copy()

LAGS = [1, 2, 3, 24, 48, 168]

for lag in LAGS:
    df_feat[f'precio_lag_{lag}h']  = df_feat['precio_eur_mwh'].shift(lag)
    df_feat[f'demanda_lag_{lag}h'] = df_feat['demanda_mw'].shift(lag)

print(f'Features de lags añadidas: {len(LAGS)*2}')
print('Ejemplo — primeros valores del lag 24h de precio:')
print(df_feat[['precio_eur_mwh', 'precio_lag_24h']].dropna().head(3))

---
## 3. Features de ventanas móviles (rolling)

Capturan la tendencia reciente y la volatilidad local.
Son especialmente útiles para que el modelo detecte regímenes de precio alto o bajo.

In [ ]:
WINDOWS = [6, 24, 168]  # 6h, 24h, 1 semana

for w in WINDOWS:
    # Media móvil
    df_feat[f'precio_roll_mean_{w}h']  = df_feat['precio_eur_mwh'].shift(1).rolling(w).mean()
    df_feat[f'demanda_roll_mean_{w}h'] = df_feat['demanda_mw'].shift(1).rolling(w).mean()
    # Desviación estándar (volatilidad)
    df_feat[f'precio_roll_std_{w}h']   = df_feat['precio_eur_mwh'].shift(1).rolling(w).std()
    # Máximo y mínimo en ventana
    df_feat[f'precio_roll_max_{w}h']   = df_feat['precio_eur_mwh'].shift(1).rolling(w).max()
    df_feat[f'precio_roll_min_{w}h']   = df_feat['precio_eur_mwh'].shift(1).rolling(w).min()

# Ratio precio actual vs media 24h (detecta si estamos en hora cara o barata)
df_feat['precio_ratio_24h'] = df_feat['precio_lag_1h'] / (df_feat['precio_roll_mean_24h'] + 1e-6)

print(f'Features rolling añadidas: {len(WINDOWS)*5 + 1}')
cols_rolling = [c for c in df_feat.columns if 'roll' in c or 'ratio' in c]
print(f'Columnas: {cols_rolling}')

---
## 4. Features de diferencias

Capturan la velocidad de cambio del precio — cuánto ha subido o bajado respecto a la hora anterior.
Muy útil para detectar arranques de picos de precio.

In [ ]:
# Diferencia hora a hora
df_feat['precio_diff_1h']  = df_feat['precio_eur_mwh'].diff(1)
df_feat['precio_diff_24h'] = df_feat['precio_eur_mwh'].diff(24)
df_feat['demanda_diff_1h'] = df_feat['demanda_mw'].diff(1)

# Diferencia porcentual (cambio relativo)
df_feat['precio_pct_1h']  = df_feat['precio_eur_mwh'].pct_change(1).replace([np.inf, -np.inf], 0)
df_feat['precio_pct_24h'] = df_feat['precio_eur_mwh'].pct_change(24).replace([np.inf, -np.inf], 0)

print('Features de diferencias añadidas: 5')

---
## 5. Features de interacción

Combinaciones de variables que capturan relaciones que los modelos lineales no detectan solos.
Justificadas por los hallazgos del EDA.

In [ ]:
# Temperatura al cuadrado — relación en U con precio y demanda (EDA sección 7)
df_feat['temperatura_c2'] = df_feat['temperatura_c'] ** 2

# Interacción temperatura × hora (el efecto del calor es mayor a mediodía)
df_feat['temp_x_hora'] = df_feat['temperatura_c'] * df_feat['hora']

# Interacción irradiación × hora (solar solo es relevante en horas de luz)
df_feat['irrad_x_hora'] = df_feat['irradiacion_wm2'] * df_feat['hora']

# Flag hora solar (10h-16h) — horas donde la solar impacta el precio
df_feat['es_hora_solar'] = df_feat['hora'].between(10, 16).astype(int)

# Interacción solar × hora_solar (activa solo cuando hay sol real)
df_feat['irrad_solar_activa'] = df_feat['irradiacion_wm2'] * df_feat['es_hora_solar']

# Semana del año — captura estacionalidad más fina que el mes
df_feat['semana_sin'] = np.sin(2 * np.pi * df_feat['semana_anio'] / 52)
df_feat['semana_cos'] = np.cos(2 * np.pi * df_feat['semana_anio'] / 52)

print('Features de interacción añadidas: 8')

---
## 6. Construcción de targets multihorizonte (t+1 … t+48)

Para predecir las próximas 48 horas creamos 48 columnas target.
Cada columna `target_precio_hX` contiene el precio X horas en el futuro.

Estrategia: **Direct Multi-Output** — un modelo por horizonte.
Alternativa posible: **Recursive** — predecir t+1 y usar esa predicción para t+2, etc.
Usaremos Direct porque es más robusto ante errores acumulativos.

In [ ]:
HORIZONTE = 48

for h in range(1, HORIZONTE + 1):
    df_feat[f'target_precio_h{h}']  = df_feat['precio_eur_mwh'].shift(-h)
    df_feat[f'target_demanda_h{h}'] = df_feat['demanda_mw'].shift(-h)

print(f'Targets creados: {HORIZONTE*2} columnas (h1 a h{HORIZONTE} para precio y demanda)')
print(f'\nShape actual: {df_feat.shape}')

---
## 7. Limpieza final, escalado y split temporal

In [ ]:
# ── Eliminar filas con NaN (causados por lags y targets) ──────────────────────
# Los primeros 168 registros tendrán NaN en los lags largos
# Los últimos 48 registros tendrán NaN en los targets
filas_antes = len(df_feat)
df_feat = df_feat.dropna()
filas_despues = len(df_feat)

print(f'Filas eliminadas por NaN: {filas_antes - filas_despues}')
print(f'Filas finales: {filas_despues:,}')
print(f'Columnas totales: {df_feat.shape[1]}')

In [ ]:
# ── Split temporal 70% train / 15% val / 15% test ─────────────────────────────
# IMPORTANTE: nunca usar split aleatorio en series temporales
# El test siempre debe ser el período más reciente

n = len(df_feat)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

df_train = df_feat.iloc[:n_train]
df_val   = df_feat.iloc[n_train:n_train + n_val]
df_test  = df_feat.iloc[n_train + n_val:]

print('Split temporal:')
print(f'  Train: {len(df_train):,} filas  ({df_train.index.min().date()} → {df_train.index.max().date()})')
print(f'  Val:   {len(df_val):,} filas  ({df_val.index.min().date()} → {df_val.index.max().date()})')
print(f'  Test:  {len(df_test):,} filas  ({df_test.index.min().date()} → {df_test.index.max().date()})')

# Visualizar el split
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df_train.index, df_train['precio_eur_mwh'], color=COLORS[0], lw=0.6, label=f'Train ({len(df_train):,})')
ax.plot(df_val.index,   df_val['precio_eur_mwh'],   color=COLORS[1], lw=0.6, label=f'Val ({len(df_val):,})')
ax.plot(df_test.index,  df_test['precio_eur_mwh'],  color=COLORS[3], lw=0.6, label=f'Test ({len(df_test):,})')
ax.set_title('Split temporal — precio eléctrico', fontweight='bold')
ax.set_ylabel('€/MWh')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend()
plt.tight_layout()
plt.savefig('data/processed/fe_01_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Definir grupos de columnas ─────────────────────────────────────────────────
TARGET_PRECIO  = [f'target_precio_h{h}'  for h in range(1, HORIZONTE+1)]
TARGET_DEMANDA = [f'target_demanda_h{h}' for h in range(1, HORIZONTE+1)]
ALL_TARGETS    = TARGET_PRECIO + TARGET_DEMANDA

# Columnas que NO se escalan (ya son binarias o cíclicas)
COLS_NO_ESCALAR = [
    'es_festivo', 'es_vispera_festivo', 'es_post_festivo', 'es_hora_solar',
    'hora_sin', 'hora_cos', 'dia_sin', 'dia_cos', 'mes_sin', 'mes_cos',
    'semana_sin', 'semana_cos', 'hora', 'dia_semana', 'mes', 'anio',
    'semana_anio', 'dias_hasta_festivo'
] + ALL_TARGETS

# Columnas de texto — eliminar para modelado
COLS_TEXTO = ['tipo_dia', 'nombre_festivo', 'anio_str'] if 'tipo_dia' in df_feat.columns else []

# Features numéricas que SÍ se escalan
COLS_ESCALAR = [
    c for c in df_feat.columns
    if c not in COLS_NO_ESCALAR + COLS_TEXTO
]

print(f'Columnas a escalar: {len(COLS_ESCALAR)}')
print(f'Columnas sin escalar: {len(COLS_NO_ESCALAR)}')
print(f'Columnas texto (eliminadas): {COLS_TEXTO}')

In [ ]:
# ── Escalar — fit SOLO sobre train, transform sobre val y test ─────────────────
# Regla de oro: el scaler nunca ve datos de val/test durante el fit

scaler_features = StandardScaler()   # Features: media 0, std 1
scaler_precio   = MinMaxScaler()     # Target precio: rango [0,1]
scaler_demanda  = MinMaxScaler()     # Target demanda: rango [0,1]

# Preparar DataFrames sin columnas de texto
def preparar(df_split):
    return df_split.drop(columns=[c for c in COLS_TEXTO if c in df_split.columns])

train = preparar(df_train)
val   = preparar(df_val)
test  = preparar(df_test)

# Fit + transform train
train_scaled = train.copy()
train_scaled[COLS_ESCALAR] = scaler_features.fit_transform(train[COLS_ESCALAR])
train_scaled[TARGET_PRECIO]  = scaler_precio.fit_transform(train[TARGET_PRECIO])
train_scaled[TARGET_DEMANDA] = scaler_demanda.fit_transform(train[TARGET_DEMANDA])

# Solo transform val y test
val_scaled = val.copy()
val_scaled[COLS_ESCALAR]     = scaler_features.transform(val[COLS_ESCALAR])
val_scaled[TARGET_PRECIO]    = scaler_precio.transform(val[TARGET_PRECIO])
val_scaled[TARGET_DEMANDA]   = scaler_demanda.transform(val[TARGET_DEMANDA])

test_scaled = test.copy()
test_scaled[COLS_ESCALAR]    = scaler_features.transform(test[COLS_ESCALAR])
test_scaled[TARGET_PRECIO]   = scaler_precio.transform(test[TARGET_PRECIO])
test_scaled[TARGET_DEMANDA]  = scaler_demanda.transform(test[TARGET_DEMANDA])

print('✅ Escalado completado')
print(f'   Train escalado: {train_scaled.shape}')
print(f'   Val escalado:   {val_scaled.shape}')
print(f'   Test escalado:  {test_scaled.shape}')

In [ ]:
# ── Resumen de features creadas ────────────────────────────────────────────────
feature_cols = [c for c in train.columns if c not in ALL_TARGETS + COLS_TEXTO]

grupos = {
    'Temporales originales': [c for c in feature_cols if c in ['hora','dia_semana','mes','anio','semana_anio']],
    'Encoding cíclico':      [c for c in feature_cols if any(x in c for x in ['sin','cos'])],
    'Calendario':            [c for c in feature_cols if any(x in c for x in ['festivo','vispera','post'])],
    'Meteorología':          [c for c in feature_cols if any(x in c for x in ['temperatura','humedad','viento','irrad','nubosidad','precipitacion'])],
    'Lags precio':           [c for c in feature_cols if 'precio_lag' in c],
    'Lags demanda':          [c for c in feature_cols if 'demanda_lag' in c],
    'Rolling precio':        [c for c in feature_cols if 'precio_roll' in c or 'precio_ratio' in c],
    'Rolling demanda':       [c for c in feature_cols if 'demanda_roll' in c],
    'Diferencias':           [c for c in feature_cols if 'diff' in c or 'pct' in c],
    'Interacciones':         [c for c in feature_cols if any(x in c for x in ['c2','x_hora','solar_activa','hora_solar'])],
}

print('='*55)
print('RESUMEN DE FEATURES')
print('='*55)
total = 0
for grupo, cols in grupos.items():
    print(f'  {grupo:<25} {len(cols):>3} features')
    total += len(cols)
print(f'  {"─"*38}')
print(f'  {"TOTAL":<25} {total:>3} features')
print(f'  {"Targets (precio+demanda)":<25} {len(ALL_TARGETS):>3} columnas')

---
## 8. Guardado de datasets y scalers

In [ ]:
# ── Guardar datasets sin escalar (para SARIMA y análisis) ─────────────────────
train.to_csv('data/processed/train.csv')
val.to_csv('data/processed/val.csv')
test.to_csv('data/processed/test.csv')

# ── Guardar datasets escalados (para XGBoost y LSTM) ─────────────────────────
train_scaled.to_csv('data/processed/train_scaled.csv')
val_scaled.to_csv('data/processed/val_scaled.csv')
test_scaled.to_csv('data/processed/test_scaled.csv')

# ── Guardar scalers (necesarios para invertir en predicción) ──────────────────
with open('data/models/scaler_features.pkl', 'wb') as f:
    pickle.dump(scaler_features, f)
with open('data/models/scaler_precio.pkl', 'wb') as f:
    pickle.dump(scaler_precio, f)
with open('data/models/scaler_demanda.pkl', 'wb') as f:
    pickle.dump(scaler_demanda, f)

# ── Guardar lista de features (para consistencia en el notebook de modelado) ──
import json
meta = {
    'feature_cols':     feature_cols,
    'target_precio':    TARGET_PRECIO,
    'target_demanda':   TARGET_DEMANDA,
    'cols_escalar':     COLS_ESCALAR,
    'cols_no_escalar':  [c for c in COLS_NO_ESCALAR if c not in ALL_TARGETS],
    'horizonte':        HORIZONTE,
    'n_train':          len(train),
    'n_val':            len(val),
    'n_test':           len(test),
    'train_start':      str(train.index.min()),
    'train_end':        str(train.index.max()),
    'test_start':       str(test.index.min()),
    'test_end':         str(test.index.max()),
}
with open('data/models/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Todo guardado:')
print('   data/processed/train.csv + val.csv + test.csv')
print('   data/processed/train_scaled.csv + val_scaled.csv + test_scaled.csv')
print('   data/models/scaler_features.pkl + scaler_precio.pkl + scaler_demanda.pkl')
print('   data/models/metadata.json')
print(f'\n✅ Notebook 03 completado. Siguiente: 04_modelado.ipynb')